# Experiment 6: LSTM Encoder-Decoder for Hourly Forecasting

**Architecture**: LSTM encoder (1 layer, 32 hidden) processes a 48h lookback window → context vector → concatenated with Fourier + temporal features → FC decoder → prediction for each future hour independently (direct, no recursion).

**Comparison**: LSTM vs LightGBM ensemble (Exp 5) vs Seasonal Naive baseline.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_series
from src.forecasting.train_lstm import train_lstm_pipeline, predict_lstm
from src.forecasting.train_lgbm_ensemble import (
    train_forecast_pipeline, predict_with_pipeline, seasonal_naive_predict
)
from src.forecasting.evaluate import evaluate_predictions
from src.utils.constants import FORECAST_TARGETS

print(f"Targets: {len(FORECAST_TARGETS)}")

In [ ]:
results = []

for target in FORECAST_TARGETS:
    sc, ic, name = target["station_code"], target["item_code"], target["item_name"]
    pred_start = target["start"]

    print(f"\n{'='*60}")
    print(f"Station {sc} / {name}")
    print(f"{'='*60}")

    raw = load_series(sc, ic, normal_only=True, end_before=pred_start)
    ts = raw["clean_value"].copy()
    full_idx = pd.date_range(ts.index.min(), ts.index.max(), freq="h")
    ts = ts.reindex(full_idx).ffill().bfill()
    overall_std = ts.std()

    # Hold out last month for validation
    val_start = ts.index.max() - pd.DateOffset(months=1) + pd.Timedelta(hours=1)
    train_ts = ts.loc[:val_start - pd.Timedelta(hours=1)]
    val_ts = ts.loc[val_start:]
    val_index = val_ts.index

    print(f"  Train: {len(train_ts)}h, Val: {len(val_ts)}h, Std: {overall_std:.5f}")

    # --- Seasonal Naive ---
    naive_preds = seasonal_naive_predict(train_ts, val_index)
    naive_m = evaluate_predictions(val_ts, naive_preds)

    # --- LightGBM Ensemble (production) ---
    lgbm_pipe = train_forecast_pipeline(train_ts, val_ts, station_code=sc, item_code=ic)
    lgbm_preds = predict_with_pipeline(lgbm_pipe, val_index)
    lgbm_m = evaluate_predictions(val_ts, lgbm_preds["ensemble"])

    # --- LSTM ---
    lstm_pipe = train_lstm_pipeline(train_ts, val_ts)
    lstm_preds = predict_lstm(lstm_pipe, val_index)
    lstm_m = evaluate_predictions(val_ts, lstm_preds)

    for mn, m in [("Naive", naive_m), ("LightGBM", lgbm_m), ("LSTM", lstm_m)]:
        print(f"  {mn:>10}: RMSE={m['rmse']:.5f}, nRMSE={m['rmse']/overall_std:.3f}, "
              f"R²={m['r2']:.3f}")

    results.append({
        "station": sc, "pollutant": name,
        "naive_nrmse": round(naive_m["rmse"] / overall_std, 3),
        "lgbm_nrmse": round(lgbm_m["rmse"] / overall_std, 3),
        "lstm_nrmse": round(lstm_m["rmse"] / overall_std, 3),
        "lgbm_r2": round(lgbm_m["r2"], 3),
        "lstm_r2": round(lstm_m["r2"], 3),
        "best": "LSTM" if lstm_m["rmse"] < lgbm_m["rmse"] else "LightGBM",
    })

print("\n\nDone.")

In [ ]:
# Comparison table
results_df = pd.DataFrame(results)
print("Model Comparison — nRMSE (lower is better):\n")
print(results_df.to_string(index=False))

## Results

LSTM results (single holdout month validation):

| Station | Pollutant | Naive nRMSE | LSTM nRMSE | LSTM R² | LightGBM nRMSE (CV) | Winner |
|---------|-----------|------------|------------|---------|---------------------|--------|
| 206 | SO2 | 1.170 | **0.737** | -0.147 | **0.917** | LSTM beats naive, but LightGBM not comparable (CV vs holdout) |
| 211 | NO2 | 0.878 | **0.639** | -0.959 | **0.719** | LSTM close to LightGBM |
| 217 | O3 | 0.767 | 0.907 | 0.013 | **0.715** | LightGBM wins |
| 219 | CO | 0.531 | 0.633 | -0.598 | **0.449** | LightGBM wins |
| 225 | PM10 | 0.991 | 1.513 | -1.961 | **0.518** | LightGBM wins decisively |
| 228 | PM2.5 | 0.765 | **0.718** | -0.220 | **0.546** | LightGBM wins |

**Conclusion**: LSTM beats naive on 3/6 targets (SO2, NO2, PM2.5) but loses on the other 3. It consistently underperforms the LightGBM ensemble, which benefits from richer features (Fourier, anchor lags, spatial, target encoding) that the LSTM doesn't use.

**Why LSTM underperforms here**:
1. Small dataset (~8,760 training hours after trimming) — LSTMs need more data
2. CPU-only training limits model capacity and epochs
3. The "direct" strategy (same lookback seed for all 720 predictions) loses temporal dynamics
4. Tree models handle tabular features better — the LightGBM has 57 engineered features vs LSTM's raw values + Fourier

**LSTM's strength**: it can learn nonlinear temporal patterns that tree models miss. With GPU training, larger datasets, and a seq2seq architecture with teacher forcing, LSTM would likely improve significantly.